In [0]:
%pip install -U --upgrade --quiet  langchain langchain-community langchainhub langchain-openai langchain-databricks databricks-langchain langgraph

dbutils.library.restartPython()

In [0]:
file_path="/Volumes/dev_agents/naval/raw/foodly/foodly_company_documents (1).txt"

# Read the entire text
with open(file_path, "r") as f:
    policy_text = f.read()
    print(policy_text)

## Data preprocessing
(Chunking and Embedding)

In [0]:
from langchain_community.document_loaders import TextLoader

# Load the document
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")

print(documents[0].metadata)
#print(documents[0].page_content[:1000])

In [0]:

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # ~800 characters per chunk
    chunk_overlap=100,   # 100 characters overlap to preserve context
    separators=["\n\n", "\n", " ", ""]

)

docs=text_splitter.split_documents(documents)

print(f"Created {len(docs)} chunks.")


for _doc in docs:
    print(_doc)
    print("-"*100)


In [0]:

# Prepare data for Delta
chunk_data = []
for i, d in enumerate(docs):
    chunk_data.append({
        "chunk_id": i + 1,
        "content": d.page_content,
        "metadata": str(d.metadata)  # store metadata as JSON-like string
    })

In [0]:
chunk_data

In [0]:
df=spark.createDataFrame(chunk_data)

In [0]:
df.display()

In [0]:
catalog="dev_agents"
schema="naval"

In [0]:
df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.foodly_docs")

In [0]:
%sql
select * from dev_agents.naval.foodly_docs

In [0]:
spark.sql("ALTER TABLE dev_agents.naval.foodly_docs SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

In [0]:
%sql
describe extended dev_agents.naval.foodly_docs

In [0]:
from dotenv import load_dotenv
import os
from typing import Literal, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt.tool_node import ToolNode, tools_condition




from databricks_langchain import (
    ChatDatabricks,
    VectorSearchRetrieverTool
)



# Initialize LLM
llm = ChatDatabricks(endpoint="databricks-gpt-oss-120b")



def call_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state['messages'])]}


# Initialize the retriever tool.
vs_tool = VectorSearchRetrieverTool(
  index_name="dev_agents.naval.foodly_index",
  tool_name="foodly_policy_document_retrieval_tool",
  num_results=2,
  tool_description="Use this tool to search the Foodly knowledge base for policies, procedures, and service-related information. It retrieves the most relevant chunks from the company’s official documentation, including refund rules, cancellation terms, delivery guidelines, loyalty program details, privacy policies, and escalation procedures"
)


llm_with_tools = llm.bind_tools([vs_tool])

builder = StateGraph(MessagesState)

builder.add_node("llm",call_llm)
builder.add_node("tools",ToolNode([vs_tool]))


builder.add_edge(START,"llm")
builder.add_conditional_edges("llm" , tools_condition)
builder.add_edge("tools","llm")


agent = builder.compile()



In [0]:
messages = agent.invoke({"messages": [HumanMessage("What are refund policies for Foodly?")]})

last_message = messages["messages"][-1].content
print(last_message)